# Python Generators

Generators provide a simple and powerful tool for creating iterators. Written like regular functions, they use the `yield` statement whenever they want to return data, suspending their execution state between calls.


### 1. What Are Generators?

A **generator** is a special type of function that produces a sequence of results lazily on-demand instead of computing them all at once and storing them in memory.

In Python, any function containing the `yield` statement automatically becomes a generator function. When invoked, it returns a generator object without running the function body immediately.

**Syntax**:
```python
def my_generator():
    yield value
```

In [1]:
# 1. First Generator Example
def simple_generator():
    print("-> Generator started")
    yield 10
    print("-> Resumed and yielding second value")
    yield 20
    print("-> Resumed and yielding final value")
    yield 30
    print("-> Generator finished")

# Calling the function returns a generator object; it does NOT run the function body yet!
gen = simple_generator()
print("Generator object created:", gen)
print("Type:", type(gen))

Generator object created: <generator object simple_generator at 0x74b9d86a7f40>
Type: <class 'generator'>


---
## 2. Generators vs. Iterators

All generators are iterators, but not all iterators are generators.

| Feature | Class-Based Iterator | Generator Function |
| :--- | :--- | :--- |
| **Implementation** | Class implementing `__iter__()` and `__next__()` | Regular function using the `yield` keyword |
| **Boilerplate Code** | High (class definition, state variables, exception raising) | Minimal (simple, clean function logic) |
| **State Management** | Handled manually using instance attributes (`self.index`) | Handled **automatically** by Python's frame execution stack |
| **Protocol Compliance** | Must manually raise `StopIteration` | Automatically raises `StopIteration` when function exits |
| **Memory Efficiency** | $O(1)$ memory | $O(1)$ memory |

In [7]:
# 2. Side-by-side comparison: Class Iterator vs Generator Function
# Approach A: Class-based Iterator
class NumberIterator:
    def __init__(self, limit):
        self.limit = limit
        self.current = 1
    def __iter__(self):
        return self
    def __next__(self):
        if self.current > self.limit:
            raise StopIteration
        val = self.current
        self.current += 1
        return val

# Approach B: Generator Function (equivalent behavior, 4 lines!)
def number_generator(limit):
    for i in range(1, limit + 1):
        yield i

ni = NumberIterator(5)
print("Class Iterator")
for _ in range(5):
    print(next(ni))

ng = number_generator(5)
print("Generator Function")
for _ in range(5):
    print(next(ng))

Class Iterator
1
2
3
4
5
Generator Function
1
2
3
4
5


### 3. The `yield` Keyword

The `yield` keyword temporarily pauses the generator function's execution and emits a value to the caller. Unlike `return`, the local variables, execution state, and instruction pointer are saved, allowing the function to resume exactly where it stopped on the next request.

**Syntax**:
```python
yield expression
```

In [8]:
# 3. How yield pauses and resumes execution
def count_up():
    print("Starting count...")
    yield "First"
    print("Continuing count...")
    yield "Second"
    print("Ending count...")
    yield "Third"

c = count_up()

print("Step 1:", next(c))
print("Step 2:", next(c))
print("Step 3:", next(c))

Starting count...
Step 1: First
Continuing count...
Step 2: Second
Ending count...
Step 3: Third


---
## 4. `yield` vs. `return`

| Characteristic | `return` | `yield` |
| :--- | :--- | :--- |
| **Function Type** | Normal function | Generator function |
| **State** | Destroys local stack frame upon exit | **Freezes and preserves** local variables and state |
| **Occurrences** | Terminates function after a single execution | Can yield multiple values over successive calls |
| **Output** | Returns a single value (or tuple of values) | Produces a stream of values one at a time |
| **Return inside Generator** | Terminates the generator (equivalent to `raise StopIteration(value)`) | Emits next value in stream |

In [9]:
# Normal function using return
def get_numbers_return():
    return [1, 2, 3]  # All items created in RAM at once

# Generator function using yield
def get_numbers_yield():
    yield 1
    yield 2
    yield 3

print("return produces:", get_numbers_return())
print("yield produces :", get_numbers_yield())

return produces: [1, 2, 3]
yield produces : <generator object get_numbers_yield at 0x74b9d86ee680>


In [11]:
# Return statement inside a generator halts iteration
def early_exit_generator():
    yield "A"
    yield "B"
    return "Stop Reason"  # Halts iteration
    yield "C"  # Unreachable!

for item in early_exit_generator():
    print("Early exit item:", item)

Early exit item: A
Early exit item: B


### 5. Creating Generators Using Generator Functions

Any function defined with `def` that contains at least one `yield` statement is recognized as a generator function. When called with arguments, it returns a generator iterator.

**Syntax**:
```python
def my_gen_func(*args, **kwargs):
    # setup logic
    yield result
```

In [12]:
# 5. Generator function generating geometric progression
def geometric_progression(start, ratio, terms):
    current = start
    for _ in range(terms):
        yield current
        current *= ratio

gp = geometric_progression(2, 3, 5)  # 2, 6, 18, 54, 162
print("Geometric progression:", list(gp))

Geometric progression: [2, 6, 18, 54, 162]


### 6. Generator Objects

When a generator function is invoked, Python creates and returns an instance of `generator` (type `<class 'generator'>`). It implements the iterator protocol (`__iter__()` and `__next__()`), plus coroutine methods (`send()`, `throw()`, and `close()`).

**Syntax**:
```python
gen_obj = gen_func()
dir(gen_obj)  # contains __iter__, __next__, send, throw, close
```

In [13]:
# 6. Inspecting a Generator Object
def sample_gen():
    yield "Hello"

gen_obj = sample_gen()
print("Object representation:", gen_obj)
print("Is iterable?         :", hasattr(gen_obj, "__iter__"))
print("Is iterator?         :", hasattr(gen_obj, "__next__"))
print("Has send()?          :", hasattr(gen_obj, "send"))
print("Has close()?         :", hasattr(gen_obj, "close"))
print("Has throw()?         :", hasattr(gen_obj, "throw"))

Object representation: <generator object sample_gen at 0x74b9d86ee500>
Is iterable?         : True
Is iterator?         : True
Has send()?          : True
Has close()?         : True
Has throw()?         : True


### 7. Generator Expressions

A **generator expression** provides a concise, comprehension-like syntax for creating generators on the fly. It uses parentheses `(...)` instead of square brackets `[...]`.

**Syntax**:
```python
gen_exp = (expression for item in iterable if condition)
```

In [14]:
# List comprehension (eager): [x ** 2 for x in range(5)]
# Generator expression (lazy): (x ** 2 for x in range(5))

squares_gen = (x ** 2 for x in range(1, 6))

print("Object:", squares_gen)
print("Type  :", type(squares_gen))

print("Iterating over generator expression:")
for sq in squares_gen:
    print(sq, end=" ")
print()

# Passing directly to reduction functions (sum, min, max) without extra parentheses
sum_of_squares = sum(x ** 2 for x in range(1, 11))
print("Sum of squares 1 to 10:", sum_of_squares)

Object: <generator object <genexpr> at 0x74b9d91176b0>
Type  : <class 'generator'>
Iterating over generator expression:
1 4 9 16 25 
Sum of squares 1 to 10: 385


## 8. Generator Expressions vs. List Comprehensions

| Dimension | List Comprehension `[...]` | Generator Expression `(...)` |
| :--- | :--- | :--- |
| **Evaluation Strategy** | Eager (computes entire collection immediately) | **Lazy** (computes elements one at a time on demand) |
| **Memory Footprint** | $O(N)$ (scales linearly with number of elements) | **$O(1)$** (constant memory regardless of size) |
| **Speed (Initial Creation)**| Slow for large $N$ (must allocate RAM and compute all) | **Instantaneous** |
| **Reusability** | Reusable indefinitely | **One-time use only** (exhausted upon iteration) |
| **Indexing / Slicing** | Fully supported (`lst[0]`, `lst[1:4]`) | **Not supported** |

In [15]:
# 8. Memory and speed benchmark: List Comprehension vs Generator Expression
import sys
import time

N = 5_000_000

# Memory comparison
t0 = time.perf_counter()
list_comp = [x * 2 for x in range(N)]
t_list = time.perf_counter() - t0

t0 = time.perf_counter()
gen_exp = (x * 2 for x in range(N))
t_gen = time.perf_counter() - t0

print(f"List Comprehension : Memory = {sys.getsizeof(list_comp):,} bytes | Creation time = {t_list:.4f}s")
print(f"Generator Expr     : Memory = {sys.getsizeof(gen_exp):,} bytes | Creation time = {t_gen:.6f}s")
print(f"Generator is {sys.getsizeof(list_comp) // sys.getsizeof(gen_exp):,}x smaller in memory!")

List Comprehension : Memory = 43,947,864 bytes | Creation time = 0.3468s
Generator Expr     : Memory = 208 bytes | Creation time = 0.000126s
Generator is 211,287x smaller in memory!


### 9. The `send()` Method (Bidirectional Communication)

Generators aren't just one-way data producers; they can act as **coroutines**. The `send(value)` method resumes the generator and sends a value *into* it, which becomes the evaluated result of the active `yield` expression.

*Note*: A newly created generator must first be primed by calling `next(gen)` or `gen.send(None)` before sending non-None values.

**Syntax**:
```python
received = yield emitted_value
# In caller:
gen.send(new_value)
```

In [18]:
# 9. send() Method: Running Average Coroutine
def running_average():
    total = 0.0
    count = 0
    average = None
    while True:
        # Pauses and yields current average, receives next input value
        val = yield average
        if val is None:
            break
        total += val
        count += 1
        average = total / count

avg_coroutine = running_average()

# 1. Prime the generator to the first yield
next(avg_coroutine)  # returns None initially

# 2. Send values into the generator
print("Sent 10 -> Current Avg:", avg_coroutine.send(10))  # 10.0
print("Sent 20 -> Current Avg:", avg_coroutine.send(20))  # 15.0
print("Sent 30 -> Current Avg:", avg_coroutine.send(30))  # 20.0
print("Sent 40 -> Current Avg:", avg_coroutine.send(40))  # 25.0

Sent 10 -> Current Avg: 10.0
Sent 20 -> Current Avg: 15.0
Sent 30 -> Current Avg: 20.0
Sent 40 -> Current Avg: 25.0


### 10. The `close()` Method

The `close()` method terminates a generator by raising a `GeneratorExit` exception inside it at the point where it was paused. If the generator does not catch it (or catches it and exits), the generator is cleanly closed. Subsequent calls to `next()` will raise `StopIteration`.

**Syntax**:
```python
gen.close()
```

In [19]:
# 10. close() Method demonstration
def stream_ticker():
    try:
        count = 1
        while True:
            yield f"Ticker #{count}"
            count += 1
    except GeneratorExit:
        print("-> GeneratorExit caught: Performing cleanup!")

ticker = stream_ticker()
print(next(ticker))
print(next(ticker))

# Close the generator prematurely
print("Closing generator...")
ticker.close()

# Calling next() after close() raises StopIteration
try:
    next(ticker)
except StopIteration:
    print("Generator is successfully closed and exhausted!")

Ticker #1
Ticker #2
Closing generator...
-> GeneratorExit caught: Performing cleanup!
Generator is successfully closed and exhausted!


### 11. The `throw()` Method

The `throw(type, value=None, traceback=None)` method raises an exception of the specified type *inside* the generator at the point where it is paused. The generator can handle the exception using standard `try-except` blocks or let it bubble up.

**Syntax**:
```python
gen.throw(ExceptionType, 'error message')
```

In [26]:
# 11. throw() Method demonstration
def resilient_worker():
    while True:
        try:
            yield "Working normally..."
        except ValueError as err:
            print(f"[RECOVERY] Caught ValueError inside generator: {err}")
        except RuntimeError as err:
            print(f"[CRITICAL] Fatal RuntimeError inside generator: {err}")
            break

worker = resilient_worker()
print(next(worker))

# Throwing an exception into the generator that it can handle and recover from
print("Injecting ValueError...")
print(worker.throw(ValueError("Corrupted packet received")))

# Next iteration continues normally
print(next(worker))


Working normally...
Injecting ValueError...
[RECOVERY] Caught ValueError inside generator: Corrupted packet received
Working normally...
Working normally...


### 12. `yield from` (Delegating to Sub-Generators)

Introduced in PEP 380 (Python 3.3), `yield from <iterable>` delegates part of the generator's operations to another generator or iterable. It establishes a transparent two-way pipe: values yielded by the sub-generator are passed directly to the caller, and calls to `send()`, `throw()`, and `close()` are forwarded directly to the sub-generator.

**Syntax**:
```python
def parent_gen():
    yield from sub_generator_or_iterable
```

In [27]:
# 12. yield from examples
# Example 1: Flattening multiple sequences cleanly
def combined_generator():
    yield from [1, 2, 3]
    yield from ("A", "B")
    yield from range(100, 103)

print("Flattened elements via 'yield from':")
print(list(combined_generator()))

Flattened elements via 'yield from':
[1, 2, 3, 'A', 'B', 100, 101, 102]


In [28]:
# Example 2: Delegating to sub-generators
def sub_task(task_id, steps):
    for step in range(1, steps + 1):
        yield f"Task {task_id} - Step {step}"

def main_orchestrator():
    yield "=== Workflow Started ==="
    yield from sub_task("A", 2)
    yield from sub_task("B", 3)
    yield "=== Workflow Completed ==="

print("\nWorkflow execution:")
for log in main_orchestrator():
    print(log)


Workflow execution:
=== Workflow Started ===
Task A - Step 1
Task A - Step 2
Task B - Step 1
Task B - Step 2
Task B - Step 3
=== Workflow Completed ===


### 13. Infinite Generators

Because generators compute items lazily without storing history in memory, they can represent unbounded or infinite streams (e.g. streaming sensor readings, continuous ID generation, infinite series) without crashing the system.

**Syntax**:
```python
def infinite_stream():
    while True:
        yield produce_value()
```

In [ ]:
# 13. Infinite ID & Timestamp Generator
import time

def auto_increment_id(start=1000):
    current = start
    while True:
        yield f"TXN_{current}"
        current += 1

id_stream = auto_increment_id(start=5001)

# Pull first 5 unique transaction IDs
print("Generated Transaction IDs:")
for _ in range(5):
    print(" -", next(id_stream))

Generated Transaction IDs:
 - TXN_5001
 - TXN_5002
 - TXN_5003
 - TXN_5004
 - TXN_5005


### 14. Memory Efficiency and Performance Benchmarks

Comparing memory scaling between eager lists and lazy generators. Generators maintain a flat, constant memory footprint ($O(1)$) regardless of whether they process 10 items or 100,000,000 items.

**Syntax**:
```python
# Flat memory usage across sequence sizes
```

In [30]:
# 14. Memory scaling verification across sizes
import sys

sizes = [1_000, 100_000, 1_000_000, 10_000_000]

print(f"{'Items (N)':<12}{'List Size (Bytes)':<20}{'Generator Size (Bytes)':<24}")
print("-" * 56)

for n in sizes:
    # Generator object size
    gen = (x for x in range(n))
    gen_size = sys.getsizeof(gen)
    
    # Calculate equivalent list size without allocating 10M list directly
    estimated_list_size = sys.getsizeof(range(n)) + (n * 8)
    
    print(f"{n:<12,}{estimated_list_size:<20,}{gen_size:<24,}")

print("\nNotice: Generator size remains CONSTANT at ~200 bytes regardless of N!")

Items (N)   List Size (Bytes)   Generator Size (Bytes)  
--------------------------------------------------------
1,000       8,048               200                     
100,000     800,048             200                     
1,000,000   8,000,048           200                     
10,000,000  80,000,048          200                     

Notice: Generator size remains CONSTANT at ~200 bytes regardless of N!


### 15. Practical Real-World Use Cases

Generators excel in backend pipelines, ETL jobs, and stream processing:
1. **Streaming large log files line-by-line**
2. **Data processing pipelines** (chaining filtering, parsing, and transformations)
3. **Paginated API record consumers**



In [31]:
# 15. Real-World Data Pipeline Architecture (Unix Pipeline Style in Python)
# Simulate streaming web server log data
mock_log_data = """2026-10-31 10:00:01 [INFO] User login: alice
2026-10-31 10:00:02 [ERROR] Database timeout on query
2026-10-31 10:00:05 [INFO] User login: bob
2026-10-31 10:00:09 [ERROR] Disk space low on /dev/sda1
2026-10-31 10:00:15 [WARNING] Memory usage above 85%
2026-10-31 10:00:20 [ERROR] Connection refused by auth_service
"""

# Stage 1: Line Reader Generator
def read_log_lines(data):
    for line in data.strip().split("\n"):
        yield line

# Stage 2: Filter Stage (Keep only ERROR records)
def filter_errors(lines):
    for line in lines:
        if "[ERROR]" in line:
            yield line

# Stage 3: Parser / Transformer Stage (Extract message payload)
def parse_error_messages(error_lines):
    for err in error_lines:
        parts = err.split("[ERROR] ")
        yield {"timestamp": parts[0].strip(), "error": parts[1].strip()}

# Compose the pipeline cleanly (like Unix pipes: log_data | grep ERROR | parse)
log_stream = read_log_lines(mock_log_data)
error_stream = filter_errors(log_stream)
parsed_stream = parse_error_messages(error_stream)

print("--- Processed Pipeline Output ---")
for record in parsed_stream:
    print(f"[{record['timestamp']}] Alert -> {record['error']}")

--- Processed Pipeline Output ---
[2026-10-31 10:00:02] Alert -> Database timeout on query
[2026-10-31 10:00:09] Alert -> Disk space low on /dev/sda1
[2026-10-31 10:00:20] Alert -> Connection refused by auth_service
